# Agency Actuals Profiling
**Owner:** Mia · **Table:** `pondolens.dev_bronze.agency_actuals` · **Coverage:** 2024–2025

**Goal:** identify Silver treatments and assess allotments vs obligations for Q2.

Bronze is unchanged; queries only create temporary views. Findings below use the saved notebook outputs. Original-source verification is still pending.

## 1. Columns and coverage
**Result:** 3,554 rows: 1,777 each for 2024 and 2025, from two CSVs. Business columns are strings; `_ingested_at` is a timestamp. Source sheets are `SUMMARY` and `SUM`; different sheet names alone are not an error.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.agency_actuals;


In [0]:
%sql
SELECT year, _source_file, source_name, source_page, COUNT(*) AS row_count,
       SUM(COUNT(*)) OVER () AS total_rows
FROM pondolens.dev_bronze.agency_actuals
GROUP BY year, _source_file, source_name, source_page
ORDER BY year, _source_file, source_page;


## 2. Prepare values for checks
Numbers are parsed in a temporary view. Dashes remain NULL; amounts retain their original units. No Bronze values are changed.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW mia_agency_profile AS
SELECT *, TRY_CAST(year AS INT) AS profile_year,
  NULLIF(TRIM(expense_class), '') AS profile_class,
  CASE WHEN TRIM(amount) RLIKE '^\\(.*\\)$'
       THEN -TRY_CAST(REGEXP_REPLACE(TRIM(amount), '[(),]', '') AS DECIMAL(28,6))
       ELSE TRY_CAST(REPLACE(TRIM(amount), ',', '') AS DECIMAL(28,6)) END AS profile_amount,
  TRY_CAST(REPLACE(REPLACE(TRIM(percent_amount), '%', ''), ',', '') AS DECIMAL(28,6)) AS profile_percent
FROM pondolens.dev_bronze.agency_actuals;


## 3. Missing values and rescued data
**Result:** no rescued rows, blanks, or literal `null` text. There are 809 amount dashes and 294 NULL amounts; the 294 percentage records explain the matching count, but confirm row-level alignment before treatment.

`expense_class` is NULL in 494 rows; some measures have no class breakdown. The 70 NULL `particulars` belong to Grand Total records. Do not drop these rows automatically.

In [0]:
%sql
SELECT column_name, COUNT(*) AS rows_checked,
 COUNT_IF(value IS NULL) AS nulls,
 COUNT_IF(TRIM(value) = '') AS blanks,
 COUNT_IF(LOWER(TRIM(value)) = 'null') AS null_text,
 COUNT_IF(TRIM(value) = '-') AS dashes,
 COUNT(DISTINCT value) AS distinct_values
FROM (SELECT STACK(15, 'source_name', CAST(source_name AS STRING),
    'source_page', CAST(source_page AS STRING),
    'source_row', CAST(source_row AS STRING),
    'year', CAST(year AS STRING),
    'category', CAST(category AS STRING),
    'particulars', CAST(particulars AS STRING),
    'row_type', CAST(row_type AS STRING),
    'budget_stage', CAST(budget_stage AS STRING),
    'expense_class', CAST(expense_class AS STRING),
    'unit_amount', CAST(unit_amount AS STRING),
    'amount', CAST(amount AS STRING),
    'percent_type', CAST(percent_type AS STRING),
    'percent_amount', CAST(percent_amount AS STRING),
    '_rescued_data', CAST(_rescued_data AS STRING),
    '_source_file', CAST(_source_file AS STRING)) AS (column_name, value)
      FROM mia_agency_profile)
GROUP BY column_name ORDER BY column_name;


In [0]:
%sql
SELECT * FROM mia_agency_profile
WHERE _rescued_data IS NOT NULL AND TRIM(_rescued_data) NOT IN ('', 'null')
LIMIT 50;


## 4. Grain and duplicates
**Result:** zero exact duplicates and zero repeated candidate keys.

Candidate grain: source file/sheet/row, year, category/entity, row type, stage, class, percentage type, and unit. This is a source-record key; cross-year entity mapping still needs review.

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM mia_agency_profile) - COUNT(*) AS duplicate_extra_rows
FROM (SELECT DISTINCT source_name, source_page, source_row, year, category, particulars, row_type, budget_stage, expense_class, unit_amount, amount, percent_type, percent_amount FROM mia_agency_profile);


In [0]:
%sql
SELECT _source_file, source_name, source_page, source_row, year, category, particulars, row_type, budget_stage, expense_class, percent_type, unit_amount, COUNT(*) AS row_count
FROM mia_agency_profile
GROUP BY _source_file, source_name, source_page, source_row, year, category, particulars, row_type, budget_stage, expense_class, percent_type, unit_amount
HAVING COUNT(*) > 1 ORDER BY row_count DESC;


## 5. Agencies, hierarchy, labels, and units
**Result:** four categories and four row types: `agency_total`, `category_total`, `line_item`, and `sub_line_item`. These levels overlap; do not sum them together.

NEDA appears in 2024 and DEPDev (formerly NEDA) in 2025. Confirm the mapping before year comparisons. `category` is a budget grouping, not a parent-department map. Labels such as ARMM also need source review.

In [0]:
%sql
SELECT category, row_type, COUNT(DISTINCT particulars) AS entity_count,
       COUNT(*) AS row_count
FROM mia_agency_profile GROUP BY category, row_type ORDER BY category, row_type;


In [0]:
%sql
SELECT DISTINCT category, particulars, row_type, year
FROM mia_agency_profile ORDER BY category, particulars, year;


In [0]:
%sql
SELECT budget_stage, expense_class, unit_amount, percent_type,
       COUNT(*) AS row_count
FROM mia_agency_profile
GROUP BY budget_stage, expense_class, unit_amount, percent_type
ORDER BY budget_stage, expense_class, unit_amount, percent_type;


In [0]:
%sql
SELECT particulars, row_type, COLLECT_SET(year) AS years_present
FROM mia_agency_profile GROUP BY particulars, row_type
HAVING COUNT(DISTINCT year) < (SELECT COUNT(DISTINCT year) FROM mia_agency_profile)
ORDER BY particulars;


## 6. Numbers and percentage scales
**Result:** zero amount parse failures and invalid years. There are 89 negative adjustment values and four negative Total Available Appropriations values in 2025; review their source meaning.

Amounts are labelled `thousand pesos`; rates are labelled `percent`. Parsed rates use the 0–100 scale, with no negative rates or rates above 100. **Twelve rate values failed parsing:** four per percentage type. Inspect their raw values; do not replace them with zero.

In [0]:
%sql
SELECT year, budget_stage, unit_amount, COUNT(*) AS rows_checked,
 COUNT_IF(profile_year IS NULL) AS invalid_year,
 COUNT_IF(amount IS NOT NULL AND LOWER(TRIM(amount)) NOT IN ('', 'null', '-')
          AND profile_amount IS NULL) AS amount_parse_failures,
 COUNT_IF(profile_amount < 0) AS negative_amounts,
 COUNT_IF(profile_amount = 0) AS zero_amounts,
 COUNT_IF(TRIM(amount) = '-') AS dash_amounts
FROM mia_agency_profile GROUP BY year, budget_stage, unit_amount
ORDER BY year, budget_stage, unit_amount;


In [0]:
%sql
SELECT percent_type, unit_amount, COUNT(*) AS row_count,
 MIN(profile_percent) AS minimum_rate, MAX(profile_percent) AS maximum_rate,
 COUNT_IF(profile_percent BETWEEN 0 AND 1) AS values_between_0_and_1,
 COUNT_IF(profile_percent > 100) AS values_above_100,
 COUNT_IF(profile_percent < 0) AS negative_rates,
 COUNT_IF(percent_amount IS NOT NULL
          AND LOWER(TRIM(percent_amount)) NOT IN ('', 'null', '-')
          AND profile_percent IS NULL) AS rate_parse_failures
FROM mia_agency_profile WHERE percent_type IS NOT NULL
GROUP BY percent_type, unit_amount ORDER BY percent_type, unit_amount;


## 7. Q2 pairs
**Result:** 510 amount pairs: 375 have available values, 132 have missing/unparsed amounts, and three have obligations above allotments. All 510 have one allotment and one obligation record.

**Query limitation:** 98 percentage rows are also labelled “Missing counterpart.” They are rate records, not missing monetary allotments. Filter pairing to monetary records before using status counts in a report.

The three flagged 2024 PS pairs are Allocation to LGUs, Special Purpose Funds total, and Current Year Budget. Each shows obligations above allotments by **0.275 thousand pesos (₱275)**; these may overlap in the hierarchy. Check the source and approved tolerance before deciding treatment.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW mia_agency_pairs AS
SELECT _source_file, source_name, source_page, source_row, year, category, particulars, row_type, profile_class, unit_amount,
 COUNT_IF(budget_stage = 'Allotments') AS allotment_rows,
 COUNT_IF(budget_stage = 'Obligations Incurred') AS obligation_rows,
 COUNT_IF(budget_stage = 'Unobligated Allotments') AS unobligated_rows,
 MAX(CASE WHEN budget_stage = 'Allotments' THEN profile_amount END) AS allotments,
 MAX(CASE WHEN budget_stage = 'Obligations Incurred' THEN profile_amount END) AS obligations,
 MAX(CASE WHEN budget_stage = 'Unobligated Allotments' THEN profile_amount END) AS published_unobligated
FROM mia_agency_profile
WHERE budget_stage IN ('Allotments', 'Obligations Incurred', 'Unobligated Allotments')
GROUP BY _source_file, source_name, source_page, source_row, year, category, particulars, row_type, profile_class, unit_amount
HAVING COUNT_IF(budget_stage IN ('Allotments', 'Obligations Incurred')) > 0;


In [0]:
%sql
SELECT *,
 CASE WHEN allotment_rows > 1 OR obligation_rows > 1 THEN 'Repeated stage'
      WHEN allotment_rows = 0 OR obligation_rows = 0 THEN 'Missing counterpart'
      WHEN allotments IS NULL OR obligations IS NULL THEN 'Missing/unparsed amount'
      WHEN allotments = 0 THEN 'Zero denominator'
      WHEN allotments < 0 OR obligations < 0 THEN 'Negative amount: review'
      WHEN obligations > allotments THEN 'Obligations exceed allotments: review'
      ELSE 'Pair available: confirm period and unit' END AS pair_status,
 CASE WHEN allotment_rows = 1 AND obligation_rows = 1
      THEN allotments - obligations END AS calculated_unobligated,
 CASE WHEN allotment_rows = 1 AND obligation_rows = 1 AND allotments > 0 AND obligations >= 0
      THEN obligations / allotments * 100 END AS calculated_obligation_rate,
 CASE WHEN allotment_rows = 1 AND obligation_rows = 1 AND unobligated_rows = 1
      THEN (allotments - obligations) - published_unobligated END AS unobligated_difference
FROM mia_agency_pairs ORDER BY year, particulars, profile_class;


## 8. Totals and published rates
**Class totals:** 102 of 1,008 output groups produced a comparison; the largest absolute difference is 0.000010 thousand pesos. This does not prove all totals reconcile. Other groups include dashes, unavailable components, percentage rows, or stages without class breakdowns.

**Obligation rates:** 94 pairs have both a parsed published rate and a calculated rate. Maximum difference: 0.049191 percentage points, consistent with one-decimal rounding but not independently confirmed. Four rates failed parsing and four have no published-rate record.

Confirm source precision before setting a reconciliation tolerance. NULL differences are not passes.

In [0]:
%sql
SELECT _source_file, source_name, source_page, source_row, year, category, particulars, row_type, budget_stage, unit_amount,
 COUNT_IF(profile_class IN ('PS', 'MOOE', 'FINEX', 'CO')) AS component_rows,
 COUNT(DISTINCT CASE WHEN profile_class IN ('PS', 'MOOE', 'FINEX', 'CO')
                     AND profile_amount IS NOT NULL THEN profile_class END) AS parsed_classes,
 COUNT_IF(profile_class = 'Total') AS total_rows,
 SUM(CASE WHEN profile_class IN ('PS', 'MOOE', 'FINEX', 'CO') THEN profile_amount END) AS component_sum,
 MAX(CASE WHEN profile_class = 'Total' THEN profile_amount END) AS published_total,
 CASE WHEN COUNT_IF(profile_class IN ('PS', 'MOOE', 'FINEX', 'CO')) = 4
            AND COUNT(DISTINCT CASE WHEN profile_class IN ('PS', 'MOOE', 'FINEX', 'CO')
                 AND profile_amount IS NOT NULL THEN profile_class END) = 4
            AND COUNT_IF(profile_class = 'Total') = 1
      THEN SUM(CASE WHEN profile_class IN ('PS', 'MOOE', 'FINEX', 'CO') THEN profile_amount END)
           - MAX(CASE WHEN profile_class = 'Total' THEN profile_amount END) END AS difference
FROM mia_agency_profile GROUP BY _source_file, source_name, source_page, source_row, year, category, particulars, row_type, budget_stage, unit_amount
ORDER BY year, particulars, budget_stage;


In [0]:
%sql
WITH rates AS (
 SELECT _source_file, source_name, source_page, source_row, year, category, particulars, row_type, percent_type, COUNT(*) AS rate_rows,
        MAX(profile_percent) AS published_rate
 FROM mia_agency_profile
 WHERE percent_type = 'Obligation rates'
 GROUP BY _source_file, source_name, source_page, source_row, year, category, particulars, row_type, percent_type
)
SELECT p.year, p.particulars, p.row_type, p.unit_amount,
       p.allotment_rows, p.obligation_rows, r.rate_rows, r.published_rate,
 CASE WHEN p.allotment_rows = 1 AND p.obligation_rows = 1
           AND p.allotments > 0 AND p.obligations >= 0 AND r.rate_rows = 1
      THEN p.obligations / p.allotments * 100 END AS calculated_percent,
 CASE WHEN p.allotment_rows = 1 AND p.obligation_rows = 1
           AND p.allotments > 0 AND p.obligations >= 0 AND r.rate_rows = 1
      THEN p.obligations / p.allotments END AS calculated_fraction
FROM mia_agency_pairs p LEFT JOIN rates r ON p._source_file <=> r._source_file AND p.source_name <=> r.source_name AND p.source_page <=> r.source_page AND p.source_row <=> r.source_row AND p.year <=> r.year AND p.category <=> r.category AND p.particulars <=> r.particulars AND p.row_type <=> r.row_type
WHERE p.profile_class = 'Total'
ORDER BY p.year, p.particulars;


## 9. Verify against originals
The 50 displayed rows are a preview, not proof of source verification. Check representative agency rows from both years plus failed rates, negative values, and the three PS discrepancies. Confirm the FY2025 reporting cutoff; its filename does not establish the period.

In [0]:
%sql
SELECT source_name, source_page, source_row, year, category, particulars,
 row_type, budget_stage, expense_class, unit_amount, amount, percent_type,
 percent_amount, _source_file
FROM mia_agency_profile ORDER BY year, particulars, budget_stage, expense_class
LIMIT 50;


## 10. Findings and Silver treatment

| Finding | Proposed treatment |
|---|---|
| 3,554 rows; no duplicates or rescued data | Preserve lineage; cast business fields in Silver. |
| 809 amount dashes; structural NULL fields | Confirm dash meaning; keep NULL separate from zero. Do not blanket-drop rows. |
| Mixed totals and detail levels | Preserve row types; use one non-overlapping level per measure. |
| NEDA/DEPDev and legacy labels | Confirm entity mapping; retain original labels. |
| Amounts labelled thousand pesos | After source confirmation, multiply by 1,000 for peso measures. |
| 12 percentage parse failures | Inspect raw values; flag invalid rates and preserve their source text. |
| 89 negative adjustments; four negative available appropriations | Verify source meaning before accepting or correcting. |
| 132 amount pairs not computable; three PS discrepancies | Resolve dash meaning and review source differences. |
| 98 percentage rows falsely flagged as missing counterparts | Restrict Q2 pairing to monetary records before reporting counts. |
| Partial total reconciliation; source preview only | Complete targeted source checks and document tolerance. |

### Assumptions to confirm
- Dashes, unclassified expense rows, and source rounding rules.
- FY2025 cutoff and comparability with FY2024.
- Entity mapping and which totals/detail rows can be combined.

### Semantic notes
- **Unobligated funds:** allotments − obligations.
- **Obligation rate:** obligations ÷ allotments × 100. Missing inputs or zero denominator → NULL.
- Overall rate uses summed obligations ÷ summed allotments for the same eligible records, not an average of rates.
- Standardize verified units in Silver; define reporting measures in Gold/semantic layer.
- Unobligated funds alone do not prove waste or poor performance.

### Conclusion
**Profiling outputs reviewed; Q2 is not final yet.** Resolve failed rates, dash meaning, period comparability, hierarchy/mapping, and source discrepancies. Correct the percentage-row pairing before publishing Q2 status counts.